# SBB Sitzbänke – Analyse

Gibt es einen Zusammenhang zwischen dem Anteil älterer Menschen in einer Gemeinde
und der Sitzbank-Versorgung am dortigen Bahnhof?

**Datenquellen:**
- [Mobiliar im Bahnhof](https://data.sbb.ch/explore/dataset/mobiliar-im-bahnhof/) – data.sbb.ch
- [Passagierfrequenz](https://data.sbb.ch/explore/dataset/passagierfrequenz/) – data.sbb.ch
- [Didok Haltestellen](https://data.sbb.ch/explore/dataset/dienststellen-gemass-opentransportdataswiss/) – data.sbb.ch
- [Anteil über 65-Jährige pro Gemeinde](https://mapexplorer.bfs.admin.ch/?obs=main&lang=de#c=indicator&i=ch_01_02_01a.pop65&i2=ch_01_02_01a.anteilpop65&s=2024&s2=2024&view=map179) – mapexplorer.bfs.admin.ch



In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import scipy.stats as stats

## 1. Daten laden & aufbereiten

In [2]:
mobiliar = pd.read_excel("mobiliar-im-bahnhof.xlsx")
freq = pd.read_excel("passagierfrequenz.xlsx")
didok = pd.read_excel("dienststellen-gemass-opentransportdataswiss.xlsx")
age = pd.read_csv("data-BzVsV.csv")

In [3]:
# Nur Sitzbänke behalten, Anzahl pro Bahnhof (BPUIC) summieren
sitzbänke = (
    mobiliar[mobiliar["Bezeichnung"] == "Sitzbank"]
    .groupby("BPUIC", as_index=False)["FLAME2"]
    .sum()
    .rename(columns={"FLAME2": "anzahl_sitzbaenke"})
)

# Passagierfrequenz und Gemeindename (aus Didok) dazumergen
# inner join: nur Bahnhöfe die in allen Datensätzen vorhanden sind
df = sitzbänke.merge(freq, left_on="BPUIC", right_on="UIC", how="inner")
df = df.merge(didok, left_on="BPUIC", right_on="number", how="inner")

# Altersdaten nach Gemeindename mergen
df = df.merge(age, left_on="municipalityName", right_on="Gemeinde", how="inner")

# Kennzahlen berechnen
# DTV_TJM_TGM = durchschnittlicher Tagesverkehr (Ein- und Aussteigend)
df["bank_pro_passagiere"] = df["anzahl_sitzbaenke"] / df["DTV_TJM_TGM"]

# Log-Transformation für den Plot – ohne Log dominieren Grossbahnhöfe die Skala
df["log_Bank_pro_Passagiere"] = np.log10(df["bank_pro_passagiere"])

print(f"Bahnhöfe im Datensatz: {len(df)}")

Bahnhöfe im Datensatz: 697


## 2. Erste Übersicht

In [4]:
cols = ["Bahnhof_Gare_Stazione", "Gemeinde", "anzahl_sitzbaenke", "DTV_TJM_TGM", "bank_pro_passagiere"]

print("Meiste Bänke pro Passagier*in:")
display(df.nlargest(10, "bank_pro_passagiere")[cols])

print("\nWenigste Bänke pro Passagier*in:")
display(df.nsmallest(10, "bank_pro_passagiere")[cols])

Meiste Bänke pro Passagier*in:


,Bahnhof_Gare_Stazione,Gemeinde,anzahl_sitzbaenke,DTV_TJM_TGM,bank_pro_passagiere
43,Courchavon,Courchavon,6,49,0.122449
601,Oppikon,Bussnang,4,49,0.081633
521,Quartino,Gambarogno,3,49,0.061224
111,Essert-Pittet,Chavornay,4,80,0.050000
356,Zweidlen,Glattfelden,5,100,0.050000
662,Bowil,Bowil,12,270,0.044444
459,Cormoret,Cormoret,3,70,0.042857
573,Triboltingen,Ermatingen,3,70,0.042857
460,Villeret,Villeret,5,120,0.041667
522,Magadino-Vira,Gambarogno,2,49,0.040816



Wenigste Bänke pro Passagier*in:


,Bahnhof_Gare_Stazione,Gemeinde,anzahl_sitzbaenke,DTV_TJM_TGM,bank_pro_passagiere
257,Zürich Oerlikon,Zürich,8,86200,0.000093
254,Zürich HB,Zürich,48,449800,0.000107
207,Lenzburg,Lenzburg,3,27600,0.000109
30,Aesch BL,Aesch (BL),1,8200,0.000122
255,Zürich Stadelhofen,Zürich,10,75200,0.000133
264,Zürich Hardbrücke,Zürich,11,52900,0.000208
528,Winterthur,Winterthur,22,103800,0.000212
92,Coppet,Coppet,2,7600,0.000263
696,Prilly-Malley,Prilly,2,7000,0.000286
119,Lausanne,Lausanne,30,97500,0.000308


In [5]:
cols_alter = ["Bahnhof_Gare_Stazione", "Gemeinde", "Anteil", "anzahl_sitzbaenke", "DTV_TJM_TGM"]

print("Bahnhöfe in Gemeinden mit höchstem Anteil über 65-Jähriger:")
display(df.nlargest(5, "Anteil")[cols_alter])

print("Bahnhöfe in Gemeinden mit tiefstem Anteil über 65-Jähriger:")
display(df.nsmallest(5, "Anteil")[cols_alter])

Bahnhöfe in Gemeinden mit höchstem Anteil über 65-Jähriger:


,Bahnhof_Gare_Stazione,Gemeinde,Anteil,anzahl_sitzbaenke,DTV_TJM_TGM
517,Locarno,Muralto,33.2,20,10800
42,Boncourt,Boncourt,29.7,4,190
496,Ambrì-Piotta,Quinto,29.2,1,280
590,Berlingen,Berlingen,28.9,5,260
497,Faido,Faido,28.6,6,280


Bahnhöfe in Gemeinden mit tiefstem Anteil über 65-Jähriger:


,Bahnhof_Gare_Stazione,Gemeinde,Anteil,anzahl_sitzbaenke,DTV_TJM_TGM
340,Glattbrugg,Opfikon,11.0,10,7400
341,Opfikon,Opfikon,11.0,4,2100
102,Denges-Echandens,Denges,11.1,3,350
113,Bavois,Bavois,11.3,1,310
343,Oberglatt ZH,Oberglatt,11.5,13,7400


## 3. HTML-Seite generieren

In [6]:
# Tabellen für die Webseite aufbereiten 
def tabelle_aufbereiten(data):
    t = data[["Bahnhof_Gare_Stazione", "anzahl_sitzbaenke", "DTV_TJM_TGM", "bank_pro_passagiere"]].copy()
    t.columns = ["Bahnhof", "Sitzbänke", "Tägliche Passagier*innen", "Bänke pro Passagier*in"]
    t["Bänke pro Passagier*in"] = t["Bänke pro Passagier*in"].map("{:.4f}".format)
    t["Tägliche Passagier*innen"] = t["Tägliche Passagier*innen"].map("{:,.0f}".format)
    return t

top_baenke = tabelle_aufbereiten(df.nlargest(10, "bank_pro_passagiere"))
bottom_baenke = tabelle_aufbereiten(df.nsmallest(10, "bank_pro_passagiere"))

# Kennzahlen für die Stat-Boxen
slope, intercept, r, p, se = stats.linregress(df["Anteil"], df["log_Bank_pro_Passagiere"])
avg_baenke = df["anzahl_sitzbaenke"].mean()
top_bahnhof = df.nlargest(1, "anzahl_sitzbaenke").iloc[0]

In [7]:
# Interaktiver Scatterplot
fig = px.scatter(
    df,
    x="Anteil",
    y="log_Bank_pro_Passagiere",
    hover_name="Bahnhof_Gare_Stazione",
    hover_data={
        "Anteil": ":.1f",
        "Gemeinde": True,
        "anzahl_sitzbaenke": True,
        "DTV_TJM_TGM": True,
        "log_Bank_pro_Passagiere": False, 
    },
    trendline="ols",
    labels={
        "Anteil": "Anteil über 65-Jährige (%)",
        "log_Bank_pro_Passagiere": "Sitzbänke pro Passagier*in (log)",
        "Gemeinde": "Gemeinde",
        "anzahl_sitzbaenke": "Anzahl Sitzbänke",
        "DTV_TJM_TGM": "Tägliche Passagier*innen",
    },
    title="Alter der Gemeinde vs. Sitzbank-Versorgung an Bahnhöfen",
    template="plotly_white"
)
fig.update_traces(selector=dict(mode="lines"), line=dict(color="red"))
plot_html = fig.to_html(full_html=False, include_plotlyjs="cdn")

In [8]:
def df_to_html(data, table_id):
    """Konvertiert einen DataFrame in einen HTML-Tabellenstring."""
    rows = "".join(
        f"<tr>{''.join(f'<td>{v}</td>' for v in row)}</tr>"
        for row in data.values
    )
    headers = "".join(f"<th>{c}</th>" for c in data.columns)
    return f'<table id="{table_id}"><thead><tr>{headers}</tr></thead><tbody>{rows}</tbody></table>'


html = f"""<!DOCTYPE html>
<html lang="de">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>SBB Sitzbänke Analyse</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #f5f5f5; color: #222; }}
  header {{ background: #eb0000; color: white; padding: 2rem 2rem 1.5rem; }}
  header h1 {{ font-size: 1.8rem; font-weight: 700; }}
  header p {{ margin-top: 0.5rem; opacity: 0.9; font-size: 1rem; }}
  .disclaimer {{ background: #fff8e1; border-left: 4px solid #f9a825; padding: 0.8rem 1.2rem; border-radius: 6px; font-size: 0.88rem; color: #555; margin-bottom: 1.5rem; }}
  main {{ max-width: 1100px; margin: 2rem auto; padding: 0 1.5rem; }}
  section {{ background: white; border-radius: 12px; padding: 1.5rem 2rem; margin-bottom: 2rem; box-shadow: 0 1px 4px rgba(0,0,0,0.08); }}
  h2 {{ font-size: 1.2rem; margin-bottom: 0.4rem; color: #eb0000; }}
  .subtitle {{ color: #666; font-size: 0.9rem; margin-bottom: 1.2rem; }}
  .narrative {{ font-size: 0.92rem; color: #444; line-height: 1.7; margin-bottom: 1.2rem; }}
  .stat-box {{ display: flex; gap: 1.5rem; margin-bottom: 1.5rem; flex-wrap: wrap; }}
  .stat {{ background: #fff5f5; border-left: 4px solid #eb0000; padding: 0.8rem 1.2rem; border-radius: 6px; flex: 1; min-width: 160px; }}
  .stat .val {{ font-size: 1.4rem; font-weight: 700; color: #eb0000; }}
  .stat .lbl {{ font-size: 0.8rem; color: #666; margin-top: 0.2rem; }}
  .two-col {{ display: grid; grid-template-columns: 1fr 1fr; gap: 1.5rem; }}
  .two-col > div {{ overflow-x: auto; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 0.78rem; }}
  th {{ background: #f0f0f0; padding: 0.5rem 0.6rem; text-align: left; font-weight: 600; border-bottom: 2px solid #ddd; }}
  td {{ padding: 0.45rem 0.6rem; border-bottom: 1px solid #eee; }}
  tr:hover td {{ background: #fafafa; }}
  @media (max-width: 700px) {{ .two-col {{ grid-template-columns: 1fr; }} }}
  footer {{ text-align: center; padding: 2rem; color: #999; font-size: 0.82rem; }}
</style>
</head>
<body>

<header>
  <h1>🪑 Analyse zu Sitzbänken an Schweizer Bahnhöfen</h1>
  <p>Welche Bahnhöfe haben die meisten Sitzbänke pro Passagier*in — und spielt das Alter der Gemeinde dabei eine Rolle?</p>
</header>

<main>

  <div class="disclaimer">
    ⚠️ <strong>Spassprojekt!</strong> Diese Analyse ist rein explorativ und aus Freude an öffentlichen Daten entstanden.
    Die Resultate dienen allen voran zur Unterhaltung. Es gibt viele Faktoren
    (Bahnhofsgrösse, Architektur, Kanton, ...) die hier nicht berücksichtigt werden.
  </div>

  <section>
    <h2>Auf einen Blick</h2>
    <p class="subtitle">Überblick</p>
    <div class="stat-box">
      <div class="stat"><div class="val">{len(df)}</div><div class="lbl">Bahnhöfe analysiert</div></div>
      <div class="stat"><div class="val">{int(df["anzahl_sitzbaenke"].sum())}</div><div class="lbl">Sitzbänke total</div></div>
      <div class="stat"><div class="val">{avg_baenke:.1f}</div><div class="lbl">Sitzbänke pro Bahnhof (Ø)</div></div>
      <div class="stat"><div class="val">{top_bahnhof["Bahnhof_Gare_Stazione"]}</div><div class="lbl">Meiste Bänke absolut ({int(top_bahnhof["anzahl_sitzbaenke"])} Stück)</div></div>
    </div>
  </section>

  <section>
    <h2>Wer sitzt und wer steht?</h2>
    <p class="narrative">
      Nicht alle Bahnhöfe sind gleich ausgestattet. Zwar haben grosse Knotenpunkte oft viele Bänke,
      jedoch kommen dort auf einzelne Sitzbänke oft tausende von Passagier*innen. Kleine Bahnhöfe auf dem Land
      schneiden hier oft besser ab. Die Tabellen unten zeigen die Extremfälle.
    </p>
    <div class="two-col">
      <div>
        <h3 style="margin-bottom:0.8rem; font-size:0.95rem;">🏆 Gemütlichste Bahnhöfe</h3>
        <p style="font-size:0.82rem; color:#888; margin-bottom:0.6rem;">Meiste Sitzbänke pro Passagier*in</p>
        {df_to_html(top_baenke, "top")}
      </div>
      <div>
        <h3 style="margin-bottom:0.8rem; font-size:0.95rem;">😬 Stehplatz-Bahnhöfe</h3>
        <p style="font-size:0.82rem; color:#888; margin-bottom:0.6rem;">Wenigste Sitzbänke pro Passagier*in</p>
        {df_to_html(bottom_baenke, "bottom")}
      </div>
    </div>
  </section>

  <section>
    <h2>Macht eine ältere Gemeinde den Bahnhof gemütlicher?</h2>
    <p class="narrative">
      Für wen sind Sitzbänke am wichtigsten? Seniorinnen und Senioren!
      Daher wollte ich untersuchen ob Bahnhöfe diesen Ansprüchen gerecht werden und habe untersucht
      ob es einen Zusammenhang zwischen dem Anteil
      über 65-Jähriger in einer Gemeinde und der Sitzbank-Versorgung am dortigen Bahnhof gibt.
    </p>
    <p class="narrative">
      Dabei lauert natürlich ein Störfaktor: Kleine Dorfbahnhöfe haben rechnerisch fast immer
      mehr Bänke pro Passagier*in als grosse Knotenpunkte. Kleine Bahnhöfe wiederum stehen meist in ländlichen Gemeinden, welche
      oft einen überdurchschnittlich hohen Anteil älterer Menschen haben. Ein Teil des
      Zusammenhangs dürfte also auf diesen indirekten Pfad zurückgehen.
    </p>
    <p class="narrative">
      Dennoch ist das Resultat erfreulich: Bahnhöfe in Gemeinden mit mehr älteren Menschen sind
      im Schnitt besser mit Sitzbänken ausgestattet — ob durch Zufall oder Planung sei dahingestellt.
      Der Zusammenhang ist schwach aber statistisch signifikant
      (r = {r:.2f}, p = {p:.3f}), und die Streuung im Plot zeigt deutlich dass es sich um eine Tendenz
      handelt, nicht um eine Gesetzmässigkeit.
    </p>
    <p class="narrative">
      Hover über einen Punkt um den Bahnhof sowie weitere Infos zu sehen. Die rote Linie zeigt die OLS-Regressionsgerade.
    </p>
    {plot_html}
  </section>

  <section>
    <h2>Methodik</h2>
    <p class="subtitle">Datenquellen und Vorgehen</p>
    <p class="narrative">
      Datensätze: <strong>Mobiliar im Bahnhof</strong>, <strong>Didok</strong> und <strong>Passagierfrequenz</strong> von
      <a href="https://data.sbb.ch" target="_blank">data.sbb.ch</a>, Altersdaten von
      <a href="https://opendata.swiss" target="_blank">opendata.swiss</a>.<br><br>
      Die Y-Achse im Plot zeigt log₁₀(Sitzbänke / Tägliche Passagier*innen) um die stark schiefe
      Verteilung zu normalisieren.
      Für die Tabellen wird die untransformierte Kennzahl verwendet.
      Nur Bahnhöfe die in allen Datensätzen vorhanden waren wurden berücksichtigt.
    </p>
  </section>

</main>

<footer>Spassprojekt · Analyse erstellt mit Python · Daten: SBB und BFS · <a href="https://github.com/mfriedlipublic/SBB_Benches" target="_blank" style="color: #999;">GitHub</a></footer>
</body>
</html>"""

with open("index.html", "w", encoding="utf-8") as f:
    f.write(html)

print("index.html erstellt")

index.html erstellt
